In [1]:
# 08_T_gibdd_normalize.py
"""
Нормализация ДТП из буфера в чистовые таблицы (Timeweb версия).
Без Supabase, только SQLAlchemy + pandas.
"""

import json
import pandas as pd
from logger_config import setup_logging
from db import read_sql, df_to_sql, execute_sql, engine
from config import CITIES

logger = setup_logging()

logger.info("=" * 60)
logger.info("НОРМАЛИЗАЦИЯ ДТП (JSON → 5 ТАБЛИЦ)")
logger.info("=" * 60)

# ============================================
# 1. ЗАГРУЗКА БУФЕРА (только нужные города)
# ============================================

cities_str = "', '".join(CITIES)
query = f"""
    SELECT id, kart_id, district_id, city, city_id, raw_data
    FROM gibdd_dtp_buffer
    WHERE city IN ('{cities_str}')
"""
df_buffer = read_sql(query)
logger.info(f"Загружено из буфера: {len(df_buffer)} записей")

if df_buffer.empty:
    logger.info("Нет данных для обработки")
    exit()

# ============================================
# 2. MERGE ПАТТЕРН (находим новые ДТП)
# ============================================

logger.info("Загружаем существующие ключи из gibdd_dtp_main")
df_existing = read_sql("SELECT kart_id, district_id FROM gibdd_dtp_main")
logger.info(f"Уже обработано ДТП: {len(df_existing)}")

df_merged = df_buffer.merge(
    df_existing[['kart_id', 'district_id']],
    on=['kart_id', 'district_id'],
    how='left',
    indicator=True
)
df_new = df_merged[df_merged['_merge'] == 'left_only'].drop(columns=['_merge'])

logger.info(f"Новых ДТП для обработки: {len(df_new)}")

if df_new.empty:
    logger.info("Новых ДТП нет")
    exit()

# ============================================
# ДАЛЬШЕ: разворот JSON, извлечение 5 таблиц, вставка
# ============================================


INFO: ============================================================
INFO: НОРМАЛИЗАЦИЯ ДТП (JSON → 5 ТАБЛИЦ)
INFO: ============================================================
INFO: Загружено из буфера: 386 записей
INFO: Загружаем существующие ключи из gibdd_dtp_main
INFO: Уже обработано ДТП: 0
INFO: Новых ДТП для обработки: 386


In [2]:
# 3. РАЗВОРАЧИВАЕМ ВСЕ JSON В ПЛОСКУЮ ТАБЛИЦУ (через apply)

def flatten_row(row):
    dtp = json.loads(row['raw_data'])
    df_tmp = pd.json_normalize(dtp)
    df_tmp['buffer_id'] = row['id']
    df_tmp['city_id'] = row['city_id']
    df_tmp['district_id'] = row['district_id']
    return df_tmp

# Применяем к каждой строке df_new
flat_dfs = df_new.apply(flatten_row, axis=1).tolist()
df_flat = pd.concat(flat_dfs, ignore_index=True)

logger.info(f"Плоская таблица: {len(df_flat)} строк, {len(df_flat.columns)} колонок")

# Колонки для main
main_columns = [
    'KartId', 'date', 'Time', 'DTP_V', 'POG', 'RAN', 'K_TS', 'K_UCH', 'emtp_number',
    'buffer_id', 'city_id', 'district_id'
]

# Колонки для place
place_columns = [
    'KartId',  # ← добавить сюда
    'District',
    'infoDtp.n_p', 'infoDtp.street', 'infoDtp.house',
    'infoDtp.k_ul', 'infoDtp.s_pog', 'infoDtp.s_pch', 'infoDtp.osv',
    'infoDtp.COORD_W', 'infoDtp.COORD_L',
    'infoDtp.ndu', 'infoDtp.sdor', 'infoDtp.OBJ_DTP',
    'buffer_id', 'city_id', 'district_id'
]

# Создаём датафреймы
df_main = df_flat[main_columns].copy()
df_place = df_flat[place_columns].copy()

print(f"main: {df_main.shape[0]} строк, {df_main.shape[1]} колонок")
print(f"place: {df_place.shape[0]} строк, {df_place.shape[1]} колонок")

INFO: Плоская таблица: 386 строк, 36 колонок


main: 386 строк, 12 колонок
place: 386 строк, 17 колонок


In [13]:
# РАЗВОРОТ ts_info (транспортные средства) — правильный надёжный способ

def extract_vehicles(row):
    ts_list = row['infoDtp.ts_info']
    if not ts_list:
        return pd.DataFrame()
    
    df = pd.json_normalize(ts_list)
    # Добавляем связь с ДТП вручную
    df['kart_id'] = row['KartId']
    df['district_id'] = row['district_id']
    df['city_id'] = row['city_id']
    df['buffer_id'] = row['buffer_id']
    return df

veh_dfs = df_flat.apply(extract_vehicles, axis=1).tolist()
df_vehicles = pd.concat(veh_dfs, ignore_index=True)

print("Колонки df_vehicles:", df_vehicles.columns.tolist())
print(f"Транспортных средств: {len(df_vehicles)}")
df_vehicles.head()


Колонки df_vehicles: ['n_ts', 'ts_s', 't_ts', 'marka_ts', 'm_ts', 'color', 'r_rul', 'g_v', 'm_pov', 't_n', 'f_sob', 'o_pf', 'ts_uch', 'kart_id', 'district_id', 'city_id', 'buffer_id']
Транспортных средств: 612


,n_ts,ts_s,t_ts,marka_ts,m_ts,color,r_rul,g_v,m_pov,t_n,f_sob,o_pf,ts_uch,kart_id,district_id,city_id,buffer_id
0,1,Осталось на месте ДТП,"В-класс (малый) до 3,9 м",PEUGEOT,Partner,Белый,С передним приводом,2010,,Технические неисправности отсутствуют,Частная собственность,Физические лица,"[{'K_UCH': 'Водитель', 'NPDD': ['Несоблюдение ...",225281941,46204,832,1
1,1,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",KIA,Прочие модели KIA,Серый,Полноприводные,2022,,Технические неисправности отсутствуют,Частная собственность,Физические лица,"[{'K_UCH': 'Пассажир', 'NPDD': ['Нет нарушений...",225264382,46204,832,2
2,2,Осталось на месте ДТП,"С-класс (малый средний, компактный) до 4,3 м",ВАЗ,ВАЗ 2112 и модификации,Иные цвета,С передним приводом,2005,,Технические неисправности отсутствуют,Частная собственность,Физические лица,"[{'K_UCH': 'Водитель', 'NPDD': ['Несоответстви...",225264382,46204,832,2
3,1,Осталось на месте ДТП,"В-класс (малый) до 3,9 м",ВАЗ,Прочие модели ВАЗ,Иные цвета,С передним приводом,2008,,Технические неисправности отсутствуют,Частная собственность,Физические лица,"[{'K_UCH': 'Водитель', 'NPDD': ['Выезд на поло...",225259795,46204,832,3
4,2,Осталось на месте ДТП,"В-класс (малый) до 3,9 м",CHANG-JIANG,Прочие модели Chang-Jiang,Серый,С передним приводом,2023,,Технические неисправности отсутствуют,Частная собственность,Физические лица,"[{'K_UCH': 'Водитель', 'NPDD': ['Нет нарушений...",225259795,46204,832,3


In [21]:
first_ts_uch = df_vehicles['ts_uch'][0]
print("Тип first_ts_uch:", type(first_ts_uch))
print("Длина first_ts_uch:", len(first_ts_uch))
print("Тип первого элемента first_ts_uch[0]:", type(first_ts_uch[0]))
print("Содержимое первого элемента:")
print(first_ts_uch[0])

Тип first_ts_uch: <class 'list'>
Длина first_ts_uch: 1
Тип первого элемента first_ts_uch[0]: <class 'dict'>
Содержимое первого элемента:
{'K_UCH': 'Водитель', 'NPDD': ['Несоблюдение условий, разрешающих движение транспорта задним ходом'], 'S_T': 'Не пострадал', 'POL': 'Мужской', 'V_ST': '13', 'ALCO': '88', 'SOP_NPDD': ['Управление ТС в состоянии наркотического опьянения', 'Несоблюдение требований ОСАГО'], 'SAFETY_BELT': 'Да', 'S_SM': 'Нет (не скрывался)', 'N_UCH': '2', 'S_SEAT_GROUP': '', 'INJURED_CARD_ID': ''}


In [ ]:
# Ручной разворот ts_uch через apply

def expand_ts_uch(row):
    """Разворачивает ts_uch для одного ТС"""
    records = []
    for uch in row['ts_uch']:
        record = uch.copy()
        # Добавляем все поля из row, кроме ts_uch
        for col in row.index:
            if col != 'ts_uch':
                record[col] = row[col]
        records.append(record)
    return records

# Применяем к каждой строке df_vehicles и разворачиваем в список
expanded = df_vehicles.apply(expand_ts_uch, axis=1).tolist()

# Склеиваем все записи
all_records = [record for sublist in expanded for record in sublist]
df_participants_veh = pd.DataFrame(all_records)

print(f"Участников в ТС: {len(df_participants_veh)}")
df_participants_veh.head()

Пешеходов/прочих: 202


,K_UCH,NPDD,S_T,POL,V_ST,ALCO,SOP_NPDD,S_SM,N_UCH,kart_id,...,infoDtp.s_pch,infoDtp.osv,infoDtp.change_org_motion,infoDtp.s_dtp,infoDtp.COORD_W,infoDtp.COORD_L,infoDtp.OBJ_DTP,buffer_id,city_id,district_id
0,2,[Нет нарушений],"Раненый, находящийся (находившийся) на амбулат...",Женский,,00,[Нет нарушений],Нет (не скрывался),1,225281941,...,Заснеженное,"В темное время суток, освещение включено",Режим движения не изменялся,880,55.802662,37.981303,[Многоквартирные жилые дома],1,832,46204
1,2,[Нахождение на проезжей части без цели её пере...,Скончался в течение 3 суток,Мужской,,00,[Нет нарушений],Нет (не скрывался),1,225225282,...,Сухое,"В темное время суток, освещение включено",Режим движения не изменялся,840,55.800347,37.998639,"[Остановка общественного транспорта, Надземный...",7,832,46204
2,3,[Нет нарушений],Скончался в течение 20 суток,Женский,,00,[Нет нарушений],Нет (не скрывался),1,225410157,...,Со снежным накатом,"В темное время суток, освещение включено",Режим движения не изменялся,740,55.755155,37.960973,[Многоквартирные жилые дома],8,832,46204
3,3,[Нет нарушений],Не пострадал,Женский,,00,[Нет нарушений],Нет (не скрывался),3,225410157,...,Со снежным накатом,"В темное время суток, освещение включено",Режим движения не изменялся,740,55.755155,37.960973,[Многоквартирные жилые дома],8,832,46204
4,2,[Переход через проезжую часть в неустановленно...,Скончался на месте ДТП до приезда скорой медиц...,Мужской,,00,[Нет нарушений],Нет (не скрывался),1,225399843,...,Обработанное противогололедными материалами,"В темное время суток, освещение включено",Движение частично перекрыто,800,55.819410,37.856365,"[Остановка общественного транспорта, Регулируе...",9,832,46204


In [32]:
# Ручной разворот uchInfo через apply

def expand_uch_info(row):
    """Разворачивает uchInfo для одного ДТП"""
    records = []
    for uch in row['infoDtp.uchInfo']:
        record = uch.copy()
        # Добавляем все поля из row, кроме uchInfo
        for col in row.index:
            if col != 'infoDtp.uchInfo':
                record[col] = row[col]
        records.append(record)
    return records

# Применяем к каждой строке df_flat
expanded = df_flat.apply(expand_uch_info, axis=1).tolist()

# Склеиваем все записи
all_records = [record for sublist in expanded for record in sublist]
df_participants_other = pd.DataFrame(all_records)

# Переименовываем KartId → kart_id для единообразия
if 'KartId' in df_participants_other.columns:
    df_participants_other = df_participants_other.rename(columns={'KartId': 'kart_id'})

print(f"Пешеходов/прочих: {len(df_participants_other)}")
df_participants_other.head()

Пешеходов/прочих: 202


,K_UCH,NPDD,S_T,POL,V_ST,ALCO,SOP_NPDD,S_SM,N_UCH,kart_id,...,infoDtp.s_pch,infoDtp.osv,infoDtp.change_org_motion,infoDtp.s_dtp,infoDtp.COORD_W,infoDtp.COORD_L,infoDtp.OBJ_DTP,buffer_id,city_id,district_id
0,2,[Нет нарушений],"Раненый, находящийся (находившийся) на амбулат...",Женский,,00,[Нет нарушений],Нет (не скрывался),1,225281941,...,Заснеженное,"В темное время суток, освещение включено",Режим движения не изменялся,880,55.802662,37.981303,[Многоквартирные жилые дома],1,832,46204
1,2,[Нахождение на проезжей части без цели её пере...,Скончался в течение 3 суток,Мужской,,00,[Нет нарушений],Нет (не скрывался),1,225225282,...,Сухое,"В темное время суток, освещение включено",Режим движения не изменялся,840,55.800347,37.998639,"[Остановка общественного транспорта, Надземный...",7,832,46204
2,3,[Нет нарушений],Скончался в течение 20 суток,Женский,,00,[Нет нарушений],Нет (не скрывался),1,225410157,...,Со снежным накатом,"В темное время суток, освещение включено",Режим движения не изменялся,740,55.755155,37.960973,[Многоквартирные жилые дома],8,832,46204
3,3,[Нет нарушений],Не пострадал,Женский,,00,[Нет нарушений],Нет (не скрывался),3,225410157,...,Со снежным накатом,"В темное время суток, освещение включено",Режим движения не изменялся,740,55.755155,37.960973,[Многоквартирные жилые дома],8,832,46204
4,2,[Переход через проезжую часть в неустановленно...,Скончался на месте ДТП до приезда скорой медиц...,Мужской,,00,[Нет нарушений],Нет (не скрывался),1,225399843,...,Обработанное противогололедными материалами,"В темное время суток, освещение включено",Движение частично перекрыто,800,55.819410,37.856365,"[Остановка общественного транспорта, Регулируе...",9,832,46204


In [ ]:
# Блок 1
import json
import pandas as pd
from logger_config import setup_logging
from db import read_sql, df_to_sql, execute_sql, engine, warmup
from config import CITIES

logger = setup_logging()

logger.info("=" * 60)
logger.info("НОРМАЛИЗАЦИЯ ДТП (JSON → 5 ТАБЛИЦ)")
logger.info("=" * 60)

# 1. Загрузка буфера
cities_str = "', '".join(CITIES)
query = f"""
    SELECT id, kart_id, district_id, city, city_id, raw_data
    FROM gibdd_dtp_buffer
    WHERE city IN ('{cities_str}')
"""


df_buffer = read_sql(query)
logger.info(f"Загружено из буфера: {len(df_buffer)} записей")

if df_buffer.empty:
    logger.info("Нет данных для обработки")
    exit()

# 2. Merge паттерн (находим новые ДТП)
logger.info("Загружаем существующие ключи из gibdd_dtp_main")
df_existing = read_sql("SELECT kart_id, district_id FROM gibdd_dtp_main")
logger.info(f"Уже обработано ДТП: {len(df_existing)}")

df_merged = df_buffer.merge(
    df_existing[['kart_id', 'district_id']],
    on=['kart_id', 'district_id'],
    how='left',
    indicator=True
)
df_new = df_merged[df_merged['_merge'] == 'left_only'].drop(columns=['_merge'])

logger.info(f"Новых ДТП для обработки: {len(df_new)}")

if df_new.empty:
    logger.info("Новых ДТП нет")
    exit()

# 3. РАЗВОРАЧИВАЕМ ВСЕ JSON В ПЛОСКУЮ ТАБЛИЦУ (через apply)

def flatten_row(row):
    dtp = json.loads(row['raw_data'])
    df_tmp = pd.json_normalize(dtp)
    df_tmp['buffer_id'] = row['id']
    df_tmp['city_id'] = row['city_id']
    df_tmp['district_id'] = row['district_id']
    return df_tmp

# Применяем к каждой строке df_new
flat_dfs = df_new.apply(flatten_row, axis=1).tolist()
df_flat = pd.concat(flat_dfs, ignore_index=True)

logger.info(f"Плоская таблица: {len(df_flat)} строк, {len(df_flat.columns)} колонок")

# Колонки для main
main_columns = [
    'KartId', 'date', 'Time', 'DTP_V', 'POG', 'RAN', 'K_TS', 'K_UCH', 'emtp_number',
    'buffer_id', 'city_id', 'district_id'
]

# Колонки для place
place_columns = [
    'KartId',  # ← добавить сюда
    'District',
    'infoDtp.n_p', 'infoDtp.street', 'infoDtp.house',
    'infoDtp.k_ul', 'infoDtp.s_pog', 'infoDtp.s_pch', 'infoDtp.osv',
    'infoDtp.COORD_W', 'infoDtp.COORD_L',
    'infoDtp.ndu', 'infoDtp.sdor', 'infoDtp.OBJ_DTP',
    'buffer_id', 'city_id', 'district_id'
]

# Создаём датафреймы
df_main = df_flat[main_columns].copy()
df_place = df_flat[place_columns].copy()

print(f"main: {df_main.shape[0]} строк, {df_main.shape[1]} колонок")
print(f"place: {df_place.shape[0]} строк, {df_place.shape[1]} колонок")

# РАЗВОРОТ ts_info (транспортные средства) — правильный надёжный способ

def extract_vehicles(row):
    ts_list = row['infoDtp.ts_info']
    if not ts_list:
        return pd.DataFrame()
    
    df = pd.json_normalize(ts_list)
    # Добавляем связь с ДТП вручную
    df['kart_id'] = row['KartId']
    df['district_id'] = row['district_id']
    df['city_id'] = row['city_id']
    df['buffer_id'] = row['buffer_id']
    return df

veh_dfs = df_flat.apply(extract_vehicles, axis=1).tolist()
df_vehicles = pd.concat(veh_dfs, ignore_index=True)

# Ручной разворот ts_uch через apply

def expand_ts_uch(row):
    """Разворачивает ts_uch для одного ТС"""
    records = []
    for uch in row['ts_uch']:
        record = uch.copy()
        # Добавляем все поля из row, кроме ts_uch
        for col in row.index:
            if col != 'ts_uch':
                record[col] = row[col]
        records.append(record)
    return records

# Применяем к каждой строке df_vehicles и разворачиваем в список
expanded = df_vehicles.apply(expand_ts_uch, axis=1).tolist()

# Склеиваем все записи
all_records = [record for sublist in expanded for record in sublist]
df_participants_veh = pd.DataFrame(all_records)

print(f"Участников в ТС: {len(df_participants_veh)}")

# Ручной разворот uchInfo через apply

def expand_uch_info(row):
    """Разворачивает uchInfo для одного ДТП"""
    records = []
    for uch in row['infoDtp.uchInfo']:
        record = uch.copy()
        # Добавляем все поля из row, кроме uchInfo
        for col in row.index:
            if col != 'infoDtp.uchInfo':
                record[col] = row[col]
        records.append(record)
    return records

# Применяем к каждой строке df_flat
expanded = df_flat.apply(expand_uch_info, axis=1).tolist()

# Склеиваем все записи
all_records = [record for sublist in expanded for record in sublist]
df_participants_other = pd.DataFrame(all_records)

# Переименовываем KartId → kart_id для единообразия
if 'KartId' in df_participants_other.columns:
    df_participants_other = df_participants_other.rename(columns={'KartId': 'kart_id'})

print(f"Пешеходов/прочих: {len(df_participants_other)}")
df_participants_other.head()

INFO: ============================================================
INFO: НОРМАЛИЗАЦИЯ ДТП (JSON → 5 ТАБЛИЦ)
INFO: ============================================================
INFO: Загружено из буфера: 386 записей
INFO: Загружаем существующие ключи из gibdd_dtp_main
INFO: Уже обработано ДТП: 0
INFO: Новых ДТП для обработки: 386
INFO: Плоская таблица: 386 строк, 36 колонок


main: 386 строк, 12 колонок
place: 386 строк, 17 колонок
Участников в ТС: 755
Пешеходов/прочих: 202


,K_UCH,NPDD,S_T,POL,V_ST,ALCO,SOP_NPDD,S_SM,N_UCH,kart_id,...,infoDtp.s_pch,infoDtp.osv,infoDtp.change_org_motion,infoDtp.s_dtp,infoDtp.COORD_W,infoDtp.COORD_L,infoDtp.OBJ_DTP,buffer_id,city_id,district_id
0,2,[Нет нарушений],"Раненый, находящийся (находившийся) на амбулат...",Женский,,00,[Нет нарушений],Нет (не скрывался),1,225281941,...,Заснеженное,"В темное время суток, освещение включено",Режим движения не изменялся,880,55.802662,37.981303,[Многоквартирные жилые дома],1,832,46204
1,2,[Нахождение на проезжей части без цели её пере...,Скончался в течение 3 суток,Мужской,,00,[Нет нарушений],Нет (не скрывался),1,225225282,...,Сухое,"В темное время суток, освещение включено",Режим движения не изменялся,840,55.800347,37.998639,"[Остановка общественного транспорта, Надземный...",7,832,46204
2,3,[Нет нарушений],Скончался в течение 20 суток,Женский,,00,[Нет нарушений],Нет (не скрывался),1,225410157,...,Со снежным накатом,"В темное время суток, освещение включено",Режим движения не изменялся,740,55.755155,37.960973,[Многоквартирные жилые дома],8,832,46204
3,3,[Нет нарушений],Не пострадал,Женский,,00,[Нет нарушений],Нет (не скрывался),3,225410157,...,Со снежным накатом,"В темное время суток, освещение включено",Режим движения не изменялся,740,55.755155,37.960973,[Многоквартирные жилые дома],8,832,46204
4,2,[Переход через проезжую часть в неустановленно...,Скончался на месте ДТП до приезда скорой медиц...,Мужской,,00,[Нет нарушений],Нет (не скрывался),1,225399843,...,Обработанное противогололедными материалами,"В темное время суток, освещение включено",Движение частично перекрыто,800,55.819410,37.856365,"[Остановка общественного транспорта, Регулируе...",9,832,46204
